In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import joblib
from pathlib import Path
import sys
from sklearn.metrics import mean_squared_error, r2_score
PROJECT_ROOT = Path.cwd().parents[0]

In [2]:
import pandas as pd

results = []

for h in [1, 2, 3, 4]:
   
    # Load files (adjust filenames as needed)
    gpf = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "feature_importance" / "standard" / f"GPFI_W_{h}_standard.csv"))
    groups = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "feature_importance" / "standard" / f"Groups_W_{h}_standard.csv"))
   
    # Make sure column names match
    gpf = gpf.rename(columns={"Group": "Group", "Mean_Importance": "Importance"})
    groups = groups.rename(columns={"Group": "Group", "Feature": "Feature"})
   
    # ----------------------------
    # Get top 3 groups
    # ----------------------------
    top_groups = gpf.sort_values("Importance", ascending=False).head(3).copy()
   
    # Add rank
    top_groups["Rank"] = [1, 2, 3]
   
    # ----------------------------
    # Merge with feature list
    # ----------------------------
    merged = groups.merge(top_groups, on="Group")
   
    # ----------------------------
    # Aggregate features per group
    # ----------------------------
    summary = (
        merged.groupby(["Group", "Importance", "Rank"], as_index=False)
        .agg({
            "Feature": lambda x: ", ".join(sorted(x)),
            "Feature": "count"
        })
    )
   
    # Rename columns
    summary = summary.rename(columns={"Feature": "Group_Size"})
   
    # Add features separately
    features = (
        merged.groupby("Group")["Feature"]
        .apply(lambda x: ", ".join(sorted(x)))
        .reset_index()
    )
   
    summary = summary.merge(features, on="Group")
   
    # Add horizon
    summary["Horizon"] = h
   
    results.append(summary)

final_df = pd.concat(results, ignore_index=True)

# Reorder columns
final_df = final_df[["Horizon", "Rank", "Group", "Importance", "Group_Size", "Feature"]]

print(final_df)

    Horizon  Rank  Group  Importance  Group_Size  \
0         1     3    108    0.064932           5   
1         1     1    184    0.155484          17   
2         1     2    230    0.076548           5   
3         2     3    172    0.010713           2   
4         2     1    380    0.016214          17   
5         2     2    441    0.013931           6   
6         3     2    223    0.010532           5   
7         3     3    251    0.009388           5   
8         3     1    287    0.010617           5   
9         4     3    177    0.009395           6   
10        4     2    200    0.009669          10   
11        4     1    234    0.010662           4   

                                              Feature  
0   h_500_weekave_pacific_ocean_pc6, h_850_weekave...  
1   W_current_week, h_500_weekave_conus_pc1, h_500...  
2   h_500_weekave_pacific_ocean_pc3, h_850_weekave...  
3   h_850_weekave_pacific_trop_pc14, slp_weekave_p...  
4   W_current_week, h_500_weekave_conus_pc1

In [4]:
# Sort nicely
final_df = final_df.sort_values(["Horizon", "Rank"]).copy()

# Round importance
final_df["Importance"] = final_df["Importance"].round(3)

# Optional: rename columns for table
latex_df = final_df.rename(columns={
    "Horizon": "Lead time",
    "Rank": "Rank",
    "Group": "Group",
    "Importance": "Importance",
    "Group_Size": "Group size",
    "Feature": "Features"
})

# Escape underscores for LaTeX
latex_df["Features"] = latex_df["Features"].str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(
    index=False,
    escape=False,   # keep \_ intact
    longtable=True, # good for long feature lists
    column_format=r"ccrlp{0.45\textwidth",
    caption="Top three most important grouped PFI predictor groups for RF forecasts in the West region across lead times.",
    label="tab:west_top_groups"
)

with open("west_top_groups_table.tex", "w") as f:
    f.write(latex_table)

print(latex_table)


\begin{longtable}{ccrlp{0.45\textwidth}
\caption{Top three most important grouped PFI predictor groups for RF forecasts in the West region across lead times.} \label{tab:west_top_groups} \\
\toprule
Lead time & Rank & Group & Importance & Group size & Features \\
\midrule
\endfirsthead
\caption[]{Top three most important grouped PFI predictor groups for RF forecasts in the West region across lead times.} \\
\toprule
Lead time & Rank & Group & Importance & Group size & Features \\
\midrule
\endhead
\midrule
\multicolumn{6}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
1 & 1 & 184 & 0.155000 & 17 & W\_current\_week, h\_500\_weekave\_conus\_pc1, h\_500\_weekave\_mexico\_gulf\_pc2, h\_500\_weekave\_southern\_canada\_pc5, h\_850\_weekave\_conus\_pc2, h\_850\_weekave\_mexico\_gulf\_pc2, h\_850\_weekave\_southern\_canada\_pc5, slp\_weekave\_conus\_pc4, slp\_weekave\_mexico\_gulf\_pc2, slp\_weekave\_southern\_canada\_pc6, t\_500\_weekave\_conus\_pc2, t\_500\_weekave\

In [2]:
from pathlib import Path
import pandas as pd
import re

BASE_DIR = Path("/home/mfholth/subseasonal/weekly_data/final_code/outputs/tuning/")  # change this
OUT_DIR = Path("/home/mfholth/subseasonal/weekly_data/final_code/notebooks/")           # change this
OUT_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = ["MW", "NE", "SE", "SW", "W"]

def infer_region(filename):
    for r in REGIONS:
        if re.search(fr"(^|[_-]){r}([_-]|\.|$)", filename):
            return r
    raise ValueError(f"Could not infer region from filename: {filename}")

def load_tuning_folder(folder_name):
    folder = BASE_DIR / folder_name
    dfs = []

    for file in sorted(folder.glob("*.csv")):
        df = pd.read_csv(file)
        df["Region"] = infer_region(file.name)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    # Replace blank/NA max_features with default p/3
    if "max_features" in out.columns:
        out["max_features"] = (
            out["max_features"]
            .replace(r"^\s*$", pd.NA, regex=True)
            .fillna("p/3")
        )

    # Put Region first if desired
    cols = ["Region"] + [c for c in out.columns if c != "Region"]
    out = out[cols]

    return out

def make_latex_table(df, caption, label, filename):
    latex_df = df.copy()

    # Escape underscores in string columns
    for col in latex_df.select_dtypes(include="object").columns:
        latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

    latex = latex_df.to_latex(
        index=False,
        escape=False,
        longtable=False,
        caption=caption,
        label=label,
        float_format="%.3f"
    )

    out_path = OUT_DIR / filename
    out_path.write_text(latex)
    return latex

# Load folders
standard_df = load_tuning_folder("standard")
weighted_df = load_tuning_folder("weighted")

# Save combined CSVs
standard_df.to_csv(OUT_DIR / "rf_tuning_standard_combined.csv", index=False)
weighted_df.to_csv(OUT_DIR / "rf_tuning_weighted_combined.csv", index=False)

# Save LaTeX tables
make_latex_table(
    standard_df,
    caption="Selected RF hyperparameters from standard RMSE-based tuning.",
    label="tab:rf_tuning_standard",
    filename="rf_tuning_standard.tex"
)

make_latex_table(
    weighted_df,
    caption="Selected RF hyperparameters from weighted RMSE-based tuning.",
    label="tab:rf_tuning_weighted",
    filename="rf_tuning_weighted.tex"
)

'\\begin{table}\n\\caption{Selected RF hyperparameters from weighted RMSE-based tuning.}\n\\label{tab:rf_tuning_weighted}\n\\begin{tabular}{lrrllrlrrr}\n\\toprule\nRegion & Horizon & Best_Score & region & mode & max_depth & max_features & min_samples_leaf & min_samples_split & n_estimators \\\\\n\\midrule\nMW & 1 & 2.943 & MW & weighted & 20 & p/3 & 1 & 10 & 400 \\\\\nMW & 2 & 3.367 & MW & weighted & 20 & p/3 & 1 & 5 & 200 \\\\\nMW & 3 & 3.428 & MW & weighted & 10 & log2 & 1 & 10 & 300 \\\\\nMW & 4 & 3.441 & MW & weighted & 20 & sqrt & 1 & 5 & 200 \\\\\nNE & 1 & 2.738 & NE & weighted & 20 & p/3 & 4 & 2 & 400 \\\\\nNE & 2 & 3.077 & NE & weighted & 10 & p/3 & 4 & 2 & 200 \\\\\nNE & 3 & 3.126 & NE & weighted & 50 & sqrt & 2 & 5 & 400 \\\\\nNE & 4 & 3.143 & NE & weighted & 20 & sqrt & 1 & 5 & 300 \\\\\nSE & 1 & 2.397 & SE & weighted & 20 & p/3 & 4 & 10 & 400 \\\\\nSE & 2 & 2.713 & SE & weighted & 20 & sqrt & 4 & 10 & 300 \\\\\nSE & 3 & 2.738 & SE & weighted & 50 & log2 & 4 & 10 & 300 \\\\\

In [4]:
RF_preds = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "predictions_2021_2024_RF" / f"RF_preds_standard_2021_2024.csv"))
eesn =  pd.read_csv(Path(PROJECT_ROOT / "notebooks"/ "test_pred_obs.csv"))


Bootstrap results

# MSE decomposition 

In [5]:
def murphy_mse_decomposition(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    o = y_true[mask]
    f = y_pred[mask]

    mu_o = np.mean(o)
    mu_f = np.mean(f)

    # ddof=0 keeps the identity consistent with mean squared error
    sigma_o = np.std(o, ddof=0)
    sigma_f = np.std(f, ddof=0)

    rho = np.corrcoef(o, f)[0, 1]

    correlation_component = sigma_o**2 * (1 - rho**2)
    conditional_bias = (sigma_f - rho * sigma_o)**2
    unconditional_bias = (mu_f - mu_o)**2

    mse = np.mean((f - o)**2)

    return {
        "MSE": mse,
        "Correlation_component": correlation_component,
        "Conditional_bias": conditional_bias,
        "Unconditional_bias": unconditional_bias,
        "rho": rho,
        "forecast_sd": sigma_f,
        "observed_sd": sigma_o,
        "sum_components": (
            correlation_component
            + conditional_bias
            + unconditional_bias
        )
    }


In [5]:
results = []

for region in sorted(eesn["Region"].unique()):
    for horizon in sorted(eesn["Horizon"].unique()):

        sub = (
            eesn[
                (eesn["Region"] == region) &
                (eesn["Horizon"] == horizon)
            ]
            .sort_values("Forecast_date")
            .dropna(subset=["Obs_detrended"])
            .dropna(subset=["Pred_detrended"])
        )

        y_true = sub["Obs_detrended"].to_numpy()
        y_pred = sub["Pred_detrended"].to_numpy()

        res = murphy_mse_decomposition(y_true, y_pred)

        # Add Region and Horizon to the returned metrics
        res["Region"] = region
        res["Horizon"] = horizon

        results.append(res)

cols = ["Region", "Horizon", "MSE", "Correlation_component", "Conditional_bias",
        "Unconditional_bias", "rho", "forecast_sd", "observed_sd", "sum_components"]

MSE_decomp_results = pd.DataFrame(results)[cols]
MSE_decomp_results.to_csv(Path(PROJECT_ROOT / "outputs" / "predictions_2021_2024_RF"/ "MSE_DECOMP_EESN.csv"), index=False)

In [6]:
results = []

for region in sorted(RF_preds["Region"].unique()):
    for horizon in sorted(RF_preds["Horizon"].unique()):

        sub = (
            RF_preds[
                (RF_preds["Region"] == region) &
                (RF_preds["Horizon"] == horizon)
            ]
            .sort_values("Forecast_date")
        )

        y_true = sub["Obs_detrended"].to_numpy()
        y_pred = sub["Pred_detrended"].to_numpy()

        res = murphy_mse_decomposition(y_true, y_pred)

        # Add Region and Horizon to the returned metrics
        res["Region"] = region
        res["Horizon"] = horizon

        results.append(res)

cols = ["Region", "Horizon", "MSE", "Correlation_component", "Conditional_bias",
        "Unconditional_bias", "rho", "forecast_sd", "observed_sd", "sum_components"]

MSE_decomp_results = pd.DataFrame(results)[cols]
MSE_decomp_results.to_csv(Path(PROJECT_ROOT / "outputs" / "predictions_2021_2024_RF"/ "MSE_DECOMP_RF.csv"), index=False)

In [7]:
display(MSE_decomp_results.style.hide(axis="index"))

Region,Horizon,MSE,Correlation_component,Conditional_bias,Unconditional_bias,rho,forecast_sd,observed_sd,sum_components
MW,1,7.804666,7.511005,0.033707,0.259954,0.489329,1.354147,3.142556,7.804666
MW,2,10.208928,9.715946,0.024223,0.468758,0.127169,0.555275,3.142556,10.208928
MW,3,10.748758,9.850611,0.291135,0.607012,-0.050359,0.381314,3.142556,10.748758
MW,4,10.711583,9.863684,0.242338,0.605561,-0.034818,0.382862,3.142556,10.711583
NE,1,4.626035,4.498409,0.001672,0.125954,0.452124,1.115984,2.377861,4.626035
NE,2,5.833973,5.550280,0.011459,0.272234,0.135585,0.429452,2.377861,5.833973
NE,3,5.854125,5.595715,0.011649,0.246761,0.101725,0.349819,2.377861,5.854125
NE,4,6.082588,5.633333,0.043875,0.405379,0.060784,0.354001,2.377861,6.082588
SE,1,3.850314,3.757848,0.029020,0.063447,0.520789,1.012236,2.270762,3.850314
SE,2,5.245521,5.055986,0.017370,0.172165,0.139522,0.448617,2.270762,5.245521


In [41]:
MSE_decomp_results.to_latex(
        index=False,
        escape=False,
        longtable=False,
        caption="MSE decomposition for RF predictions.",
        label="tab:MSE_decomp",
        float_format="%.3f"
    )

'\\begin{table}\n\\caption{MSE decomposition for RF predictions.}\n\\label{tab:MSE_decomp}\n\\begin{tabular}{lrrrrrrrrr}\n\\toprule\nRegion & Horizon & MSE & Correlation_component & Conditional_bias & Unconditional_bias & rho & forecast_sd & observed_sd & sum_components \\\\\n\\midrule\nMW & 1 & 7.229 & 6.835 & 0.075 & 0.319 & 0.524 & 1.334 & 3.069 & 7.229 \\\\\nMW & 2 & 9.017 & 8.335 & 0.046 & 0.636 & 0.114 & 0.546 & 2.906 & 9.017 \\\\\nMW & 3 & 9.536 & 8.463 & 0.285 & 0.788 & -0.051 & 0.386 & 2.913 & 9.536 \\\\\nMW & 4 & 9.373 & 8.513 & 0.085 & 0.776 & 0.028 & 0.372 & 2.919 & 9.373 \\\\\nNE & 1 & 4.665 & 4.510 & 0.001 & 0.154 & 0.454 & 1.109 & 2.384 & 4.665 \\\\\nNE & 2 & 5.854 & 5.509 & 0.022 & 0.322 & 0.120 & 0.431 & 2.364 & 5.854 \\\\\nNE & 3 & 5.858 & 5.553 & 0.013 & 0.292 & 0.101 & 0.352 & 2.369 & 5.858 \\\\\nNE & 4 & 6.080 & 5.571 & 0.010 & 0.498 & 0.102 & 0.344 & 2.373 & 6.080 \\\\\nSE & 1 & 3.842 & 3.735 & 0.034 & 0.073 & 0.529 & 1.018 & 2.277 & 3.842 \\\\\nSE & 2 & 5.119 & 4

In [30]:
RF_RMSE_o = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "RMSE_results_2021_2024_RF" / f"RF_rmse_results_standard_2021_2024.csv"))
RF_RMSE_w = pd.read_csv(Path(PROJECT_ROOT / "outputs"/ "RMSE_results_2021_2024_RF" / f"RF_rmse_results_weighted_2021_2024.csv"))

In [18]:
RF_RMSE_o = RF_RMSE_o[['Region', 'Horizon', 'RMSE', 'R_squared']]
RF_RMSE_w = RF_RMSE_w[['Region', 'Horizon', 'Weighted_RMSE_smooth', 'R_squared']]

In [19]:
RF_RMSE = RF_RMSE_o.copy()
RF_RMSE['Weighted_RMSE_smooth'] = RF_RMSE_w['Weighted_RMSE_smooth']

In [21]:
RF_RMSE = RF_RMSE[['Region', 'Horizon', 'RMSE', 'Weighted_RMSE_smooth', 'R_squared']].rename(columns={'Weighted_RMSE_smooth': 'wRMSE'})

In [29]:
RF_RMSE.to_csv(Path(PROJECT_ROOT / "notebooks"/ "RMSE_R_SQ_RESULTS.csv"), float_format="%.2f", index=False)

In [31]:
print(RF_RMSE_o)



   Region  Horizon      RMSE  RMSE_extreme  Weighted_RMSE_step  \
0       W        1  2.278535      2.744607            2.409578   
1      SW        1  2.059788      2.822545            2.269369   
2      MW        1  2.793683      3.729324            3.056377   
3      SE        1  1.962222      2.621433            2.145457   
4      NE        1  2.150822      2.744615            2.319742   
5       W        2  2.455600      3.073534            2.631831   
6      SW        2  2.251484      3.105006            2.486575   
7      MW        2  3.195141      4.273511            3.498129   
8      SE        2  2.290310      3.044759            2.499629   
9      NE        2  2.415362      3.271244            2.664277   
10      W        3  2.396388      3.036640            2.579821   
11     SW        3  2.259494      3.102816            2.491400   
12     MW        3  3.278530      4.376910            3.586925   
13     SE        3  2.309985      3.102151            2.530610   
14     NE 